# Web Scraping and Text Preprocessing

This project collects paragraph text from two web pages and applies a basic NLP preprocessing pipeline.

## Source pages
- Wikipedia: New Cairo
- RMS Titanic Inc.: Research and Conservation

## Workflow
1. Send HTTP requests and parse the returned HTML with BeautifulSoup.
2. Extract paragraph text from the pages.
3. Convert text to lowercase.
4. Remove numbers, punctuation, and special characters.
5. Tokenize the cleaned text using NLTK.
6. Remove English stopwords and identify unique tokens.



## 1. Imports and configuration

In [1]:
import re
import time
from pathlib import Path

import requests
import pandas as pd
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

URLS = {
    "New Cairo - Wikipedia": "https://en.wikipedia.org/wiki/New_Cairo",
    "RMS Titanic Inc. - Research and Conservation": (
        "https://rmstitanicinc.com/research-and-conservation/"
    ),
}

REQUEST_TIMEOUT = 20
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; NLP-Portfolio-Project/1.0)"
}

## 2. Scrape paragraph text

In [2]:
def scrape_paragraphs(url, source_name):
    """Fetch a page and return its non-empty paragraph texts."""
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"Could not retrieve {source_name}: {error}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove non-content elements before extracting paragraphs.
    for element in soup(["script", "style", "noscript"]):
        element.decompose()

    paragraphs = [
        paragraph.get_text(" ", strip=True)
        for paragraph in soup.find_all("p")
    ]
    paragraphs = [text for text in paragraphs if text]

    print(f"{source_name}: extracted {len(paragraphs)} paragraphs")
    return paragraphs


records = []
for source_name, url in URLS.items():
    paragraphs = scrape_paragraphs(url, source_name)
    for paragraph_number, paragraph in enumerate(paragraphs, start=1):
        records.append({
            "source": source_name,
            "url": url,
            "paragraph_number": paragraph_number,
            "raw_text": paragraph
        })

raw_df = pd.DataFrame(records)

if raw_df.empty:
    raise RuntimeError(
        "No paragraphs were extracted. Check your internet connection "
        "or whether the source websites changed their page structure."
    )

print("Total extracted paragraphs:", len(raw_df))
display(raw_df.head())

New Cairo - Wikipedia: extracted 20 paragraphs
RMS Titanic Inc. - Research and Conservation: extracted 34 paragraphs
Total extracted paragraphs: 54


,source,url,paragraph_number,raw_text
0,New Cairo - Wikipedia,https://en.wikipedia.org/wiki/New_Cairo,1,New Cairo [ a ] is a modern satellite city wit...
1,New Cairo - Wikipedia,https://en.wikipedia.org/wiki/New_Cairo,2,"Administratively, it is officially part of the..."
2,New Cairo - Wikipedia,https://en.wikipedia.org/wiki/New_Cairo,3,The city was established in 2000 as part of Eg...
3,New Cairo - Wikipedia,https://en.wikipedia.org/wiki/New_Cairo,4,According to the 2026 Prime Minister Mostafa M...
4,New Cairo - Wikipedia,https://en.wikipedia.org/wiki/New_Cairo,5,"After the 1952 revolution , in which the pro-W..."


## 3. Inspect the collected text

In [3]:
print("Number of source pages:", raw_df["source"].nunique())
print("Total characters:", raw_df["raw_text"].str.len().sum())
print("Paragraphs per source:")
display(raw_df.groupby("source").size().rename("paragraph_count").to_frame())

print("\\nSample raw paragraph:")
print(raw_df.loc[0, "raw_text"])

Number of source pages: 2
Total characters: 22492
Paragraphs per source:


,paragraph_count
source,
New Cairo - Wikipedia,20
RMS Titanic Inc. - Research and Conservation,34


\nSample raw paragraph:
New Cairo [ a ] is a modern satellite city within the Cairo Governorate of Egypt and the metropolitan area of Greater Cairo , whose formation was first initiated by a presidential decree in 1995 during the presidency of former Egyptian President Hosni Mubarak . [ 2 ]


## 4. Lowercasing

In [4]:
def lowercase_text(text):
    return text.lower()

raw_df["lower_text"] = raw_df["raw_text"].apply(lowercase_text)

print(raw_df[["raw_text", "lower_text"]].head(3).to_string(index=False))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              raw_text                                                                                                                                                                                                                                                                                                                  

## 5. Remove numbers, punctuation, and special characters

In [5]:
def clean_text(text):
    """Keep English letters and whitespace, then normalize whitespace."""
    text = re.sub(r"\\d+", " ", text)
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

raw_df["clean_text"] = raw_df["lower_text"].apply(clean_text)
raw_df = raw_df[raw_df["clean_text"].str.len() > 0].reset_index(drop=True)

print(raw_df[["lower_text", "clean_text"]].head(3).to_string(index=False))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            lower_text                                                                                                                                                                                                                                                                                                                  

## 6. Tokenization

In [6]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

raw_df["tokens"] = raw_df["clean_text"].apply(word_tokenize)

all_tokens = [
    token
    for paragraph_tokens in raw_df["tokens"]
    for token in paragraph_tokens
]

print("Total tokens before stopword removal:", len(all_tokens))
print("Sample tokens:", all_tokens[:40])

Total tokens before stopword removal: 3381
Sample tokens: ['new', 'cairo', 'a', 'is', 'a', 'modern', 'satellite', 'city', 'within', 'the', 'cairo', 'governorate', 'of', 'egypt', 'and', 'the', 'metropolitan', 'area', 'of', 'greater', 'cairo', 'whose', 'formation', 'was', 'first', 'initiated', 'by', 'a', 'presidential', 'decree', 'in', 'during', 'the', 'presidency', 'of', 'former', 'egyptian', 'president', 'hosni', 'mubarak']


## 7. Stopword removal

In [7]:
nltk.download("stopwords", quiet=True)
english_stopwords = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token not in english_stopwords
    ]

raw_df["filtered_tokens"] = raw_df["tokens"].apply(remove_stopwords)

filtered_tokens = [
    token
    for paragraph_tokens in raw_df["filtered_tokens"]
    for token in paragraph_tokens
]

unique_words = sorted(set(filtered_tokens))

print("Tokens after stopword removal:", len(filtered_tokens))
print("Number of unique words:", len(unique_words))
print("Sample filtered tokens:", filtered_tokens[:40])
print("Sample unique words:", unique_words[:40])

Tokens after stopword removal: 2067
Number of unique words: 1051
Sample filtered tokens: ['new', 'cairo', 'modern', 'satellite', 'city', 'within', 'cairo', 'governorate', 'egypt', 'metropolitan', 'area', 'greater', 'cairo', 'whose', 'formation', 'first', 'initiated', 'presidential', 'decree', 'presidency', 'former', 'egyptian', 'president', 'hosni', 'mubarak', 'administratively', 'officially', 'part', 'eastern', 'area', 'cairo', 'city', 'like', 'new', 'settlements', 'egypt', 'directly', 'governed', 'new', 'urban']
Sample unique words: ['abdel', 'accelerated', 'accelerating', 'accessibility', 'accommodates', 'according', 'accuracy', 'acres', 'across', 'actively', 'activity', 'addition', 'adequate', 'administered', 'administration', 'administrations', 'administrative', 'administratively', 'admission', 'advanced', 'affairs', 'affordability', 'affordable', 'agencies', 'agency', 'ages', 'agricultural', 'ahly', 'aimed', 'air', 'akin', 'al', 'alex', 'alleviate', 'alleviating', 'alliance', 'al

## 8. Compare text statistics

In [8]:
statistics = pd.DataFrame({
    "stage": ["Raw text", "After cleaning", "After stopword removal"],
    "total_tokens": [
        raw_df["raw_text"].apply(lambda text: len(word_tokenize(text))).sum(),
        len(all_tokens),
        len(filtered_tokens)
    ],
    "unique_tokens": [
        len(set(
            token.lower()
            for text in raw_df["raw_text"]
            for token in word_tokenize(text)
            if token.isalpha()
        )),
        len(set(all_tokens)),
        len(set(filtered_tokens))
    ]
})

display(statistics)

,stage,total_tokens,unique_tokens
0,Raw text,4042,1101
1,After cleaning,3381,1135
2,After stopword removal,2067,1051


## 9. Save processed data

In [10]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

export_df = raw_df.copy()
export_df["tokens"] = export_df["tokens"].apply(lambda values: " ".join(values))
export_df["filtered_tokens"] = export_df["filtered_tokens"].apply(
    lambda values: " ".join(values)
)
export_df.to_csv(output_dir / "preprocessed_paragraphs.csv", index=False)

pd.DataFrame({"token": unique_words}).to_csv(
    output_dir / "unique_words.csv", index=False
)

statistics.to_csv(output_dir / "preprocessing_statistics.csv", index=False)

print(f"Saved outputs to: {output_dir.resolve()}")

Saved outputs to: /content/outputs


## 10. Key observations and limitations

- The notebook extracts paragraph elements (`<p>`) only; page navigation, tables, and other HTML elements are not included.
- Web pages can change their structure or restrict automated requests, so scraping may fail even when a URL is still accessible.
- The cleaning rule keeps English letters only. It removes digits and non-English characters, matching the scope of this assignment.
- Stopword removal is based on NLTK's English stopword list.
- The project demonstrates preprocessing and does not perform sentiment analysis, classification, or model training.
- Respect each website's terms of use, robots.txt guidance, and rate limits when scraping.